# Build 03-05 · Re-evaluation — operating points per axis on the OOT split

**Kernel: the analysis `.venv`** (`python3`) — reads parquet + CSV only, loads no model.

Compares the retrained axes (and the baseline, when its `scores` kind exists) on the version's
**OOT split**, on two slices reported side by side:
- **§3 — garage rows only** (`decision = 0`, the verified slice; scrapped rows carry forced
  labels and cannot score precision): the thesis's reading.
- **§3b — the whole split** (every row of the `targets` kind with its recorded label — scrapped
  rows forced to 1 and untreated rows included): the **company convention**, the number Allianz
  itself reads. The gap between the two is the forced-label inflation.
- **§3c — naive vs baseline at one fixed cutoff**: a population-loss diagnostic, not a scheme
  comparison (see §5).

Decisions use the **STRICT rule `score > τ`** (`score == τ` garages), matching `threshold.apply`.

τ sources per model:
- **grid** — the production rule's own value(s) (`config.DECISION_RULES`): the "what if the
  production cutoff were kept" reading. A retrained model's score scale is its own, so these
  rows are reference, not the comparable operating point.
- **tuned** — `threshold.tune` on **train-split** garage rows: lowest cutoff with precision ≥
  target, the precision-floor mode of the company's own `select_best_threshold` (adopted into
  `tune()` 2026-09-02). This is the comparable operating point.
- **tuned_oot** — the same rule on the **eval-split** garage rows themselves. Train is the
  retrained model's own fitting data, so `tuned` is in-sample for the *fit*; `tuned_oot` is
  in-sample for the *evaluation* (its precision holds by construction). Neither is a clean
  held-out — the τ gap between them is the transfer drift, reported in §3.
- **tuned_all** / **tuned_oot_all** (§3b only) — the same two rules on the **whole split**,
  forced labels in: the company's own procedure end to end.

```
mitigation/inputs/corrector_targets_<v>_<oot> (03_01)   reeval/<v>_mitigated_scores_<oot>_<tag>
   ──▶  §3 metrics per (model, τ): AUC · ψ · precision · recall · TN/FP/FN/TP · scrap rate
          └▶ τ per source (tuned vs tuned_oot = drift)
          └▶ reeval/<v>_decisions_<oot>_<model>_tau<τ>.parquet   (one file per τ — 여러 갈래)
          └▶ reeval/<v>_0305_sec3_reweight_metrics_<oot>.csv
          └▶ reeval/<v>_0305_sec3_tau_by_source_<oot>.csv
   ──▶  §3b the same, on the whole split (company convention)
          └▶ reeval/<v>_0305_sec3b_reweight_metrics_<oot>_all.csv
          └▶ reeval/<v>_0305_sec3b_precision_gap_<oot>.csv
          └▶ reeval/<v>_0305_sec3b_tau_by_source_<oot>_all.csv
   ──▶  §3c naive vs baseline at one fixed cutoff (population-loss diagnostic)
          └▶ figures/mitigation/03_05/<VERSION>/<v>_0305_sec3c_naive_vs_baseline_{garage,all}.csv
   ──▶  §4 / §4b PR curves, garage slice / whole split (figstyle house style)
          ──▶ figures/mitigation/03_05/<VERSION>/
```

Every CSV this notebook writes carries `0305` (this notebook) and its section (`sec3`, `sec3b`,
`sec3c`) in the filename, matching 03_01–03_04's naming convention.

### 실행 전 설정

**커널**: analysis `.venv` (`python3`)

**바꿔야 할 것 (§1)**:
- `VERSION` — `"v2"` 또는 `"v3"`

**선행 조건**: `03_01_corrector_inputs.ipynb`(그 버전의 `corrector_targets`)와
`03_03_retrain.ipynb`(그 버전의 `reeval_scores`)가 이미 실행돼 있어야 합니다.

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle
import threshold
from detector.algorithm.residual_peak import peak0, residual

figstyle.apply()
# FIG_DIR itself is set in §1, once VERSION is known -- this notebook is rerun per VERSION
# (change §1's VERSION and rerun), so the redirect needs that value first.
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC
VERSION = "v3"        # "v2" works too once 03_01 built its corrector_targets
TRAIN_SPLIT = "train"                             # feeds "tuned"; EVAL_SPLIT feeds "tuned_oot"
EVAL_SPLIT = config.OOT_SPLIT[VERSION]            # never picked by name — v1/v2 invert "test"
ID_COL = "claim_id"
SCORE_COL = "model_" + VERSION + "_score"
TARGET_PREC = config.TARGET_PRECISION
TUNE_TAU = True       # tunes on BOTH the train split ("tuned") and the eval split ("tuned_oot")

# grid = the production rule's own cutoff value(s); override with an explicit list if needed
_rule = config.DECISION_RULES[VERSION]
if _rule["shape"] == "global":
    TAU_GRID = [float(_rule["threshold"])]
elif _rule["shape"] == "piecewise_global":
    TAU_GRID = sorted({float(r["threshold"]) for r in _rule["regimes"]})
else:
    raise SystemExit("segmented rule (v1) is out of scope here")

# §3c's fixed reference cutoff for the naive-vs-baseline population check — ONE value, not a
# tuned one, so that check is about the population, not about re-tuning. v3's grid has exactly
# one value (0.984, the real global cutoff — src/threshold.py); v2's grid has three regime
# values and 0.872 is the one CLAUDE.md documents as THE business threshold (also 00_SHAP.ipynb
# §3's stand-in) — not "the" v2 threshold in a per-row sense (five regimes alternate over time,
# see threshold.py), but the fixed reference this specific check needs.
HEADLINE_TAU = {"v2": 0.872, "v3": float(_rule["threshold"]) if _rule["shape"] == "global" else None}[VERSION]
assert any(abs(HEADLINE_TAU - t) < 1e-9 for t in TAU_GRID), (
    f"HEADLINE_TAU={HEADLINE_TAU} is not one of {VERSION}'s grid values {TAU_GRID} — fix it above")

REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval"

# Every figure/table this notebook saves is a mitigation comparison for THIS version (TRAIN_SPLIT
# + EVAL_SPLIT are both used together in one run -- EVAL_SPLIT is derived from VERSION, never an
# independent rerun knob), so the redirect keys on VERSION only: a rerun with a different VERSION
# must not collide with this one's files.
figstyle.FIG_DIR = ROOT / "figures" / "mitigation" / "03_05" / VERSION
print("eval split:", EVAL_SPLIT, "| tau grid:", TAU_GRID, "| headline tau:", HEADLINE_TAU)
print("FIG_DIR =", figstyle.FIG_DIR)

In [ ]:
# §2 — assemble: corrector_targets (treatment pre-joined by 03_01) + one score column per model
def load_split_frame(split: str) -> pd.DataFrame:
    """corrector_targets of `split` — targets already joined to the recorded treatment (03_01)."""
    p = config.split_path("corrector_targets", VERSION, split)
    assert p.is_file(), f"{p} missing — run 03_01_corrector_inputs first"
    m = pd.read_parquet(p)
    m["date"] = pd.to_datetime(m["date"])
    print(f"{VERSION} {split}: {len(m):,} treated rows (unmatched already dropped in 03_01)")
    return m


def attach_scores(frame: pd.DataFrame, split: str) -> tuple[pd.DataFrame, list[str]]:
    """Merge every available model's scores on `split`: mitigated axes + baseline if exported."""
    models = []
    bpath = config.path("scores", VERSION, split=split)
    if bpath.is_file():
        b = pd.read_parquet(bpath)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: "baseline"})
        frame = frame.merge(b, on=ID_COL, how="left")
        models.append("baseline")
    else:
        print(f"  note: no baseline scores at {bpath} (run src/scoring/score_all.py) — skipped")
    prefix = f"{VERSION}_mitigated_scores_{split}_"
    for p in sorted(REEVAL_DIR.glob(prefix + "*.parquet")):
        tag = p.stem[len(prefix):]
        s = pd.read_parquet(p)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: tag})
        frame = frame.merge(s, on=ID_COL, how="left")
        models.append(tag)
    return frame, models


ev, MODELS = attach_scores(load_split_frame(EVAL_SPLIT), EVAL_SPLIT)
tr, tr_models = attach_scores(load_split_frame(TRAIN_SPLIT), TRAIN_SPLIT)
assert MODELS, "no score files for the eval split — run 03_04 first"

matched = ev["decision"].notna()
ev_gar = ev.loc[matched & (ev["decision"] == 0)]
tr_gar = tr.loc[tr["decision"].notna() & (tr["decision"] == 0)]
# τ tuning slices keyed by tau_source: train garage (in-sample for the fit) and eval garage
# (in-sample for the evaluation). Each entry = (frame, models that have scores on it).
TUNE_GARAGE = {"tuned": (tr_gar, tr_models), "tuned_oot": (ev_gar, MODELS)}
print(f"eval garage rows: {len(ev_gar):,} ({int(ev_gar['observed'].sum()):,} verified TL) | "
      f"unmatched (no treatment): {int((~matched).sum()):,} — excluded from garage metrics")
print("models:", MODELS)

In [ ]:
# §3 — metrics per (model, τ): decisions parquet per τ + one CSV
def metrics_at(y: np.ndarray, s: np.ndarray, tau: float) -> dict:
    """AUC + precision/recall/CM at the STRICT rule score > tau (score == tau garages)."""
    pred = s > tau
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    tn = int((~pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
    return {"AUC": float(roc_auc_score(y, s)),
            "precision": tp / (tp + fp) if tp + fp else float("nan"),
            "recall": tp / (tp + fn) if tp + fn else float("nan"),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}


def recall_at_precision(y: np.ndarray, s: np.ndarray, target: float) -> float:
    """Max recall over all cutoffs while precision >= target; NaN if unreachable."""
    prec, rec, _ = precision_recall_curve(y, s)
    ok = prec >= target
    return float(rec[ok].max()) if ok.any() else float("nan")


rows = []
for name in MODELS:
    g = ev_gar[[ID_COL, "observed", name]].dropna(subset=[name])
    if len(g) < len(ev_gar):
        print(f"  {name}: {len(ev_gar) - len(g)} eval garage rows unscored — dropped for this model")
    y = g["observed"].astype(int).to_numpy()
    s = g[name].to_numpy(dtype=float)
    psi = peak0(residual(y, s))                       # threshold-free loop signal, per model
    r_tp = recall_at_precision(y, s, TARGET_PREC)

    taus = [(t, "grid") for t in TAU_GRID]
    if TUNE_TAU:
        for src, (frame, avail) in TUNE_GARAGE.items():
            if name not in avail:
                print(f"  {name}: no scores on the {src} slice — {src} tau skipped")
                continue
            tg = frame[["observed", name]].dropna(subset=[name])
            t = threshold.tune(tg["observed"].astype(int).to_numpy(),
                               tg[name].to_numpy(dtype=float), target_precision=TARGET_PREC)
            if t is None:
                print(f"  {name}: no cutoff reaches precision>={TARGET_PREC} on the {src} slice — no {src} tau")
            else:
                taus.append((float(t), src))

    all_scored = ev[[ID_COL, name]].dropna(subset=[name])
    s_all = all_scored[name].to_numpy(dtype=float)
    for tau, src in taus:
        dp = REEVAL_DIR / f"{VERSION}_decisions_{EVAL_SPLIT}_{name}_tau{round(float(tau), 6)}.parquet"
        pd.DataFrame({ID_COL: all_scored[ID_COL].to_numpy(), "score": s_all,
                      "decision": (s_all > tau).astype(int)}).to_parquet(dp, index=False)
        rows.append({"model": name, "tau": round(float(tau), 6), "tau_source": src,
                     "n_eval_garage": len(g), **metrics_at(y, s, float(tau)),
                     "psi_loop": round(psi, 3),
                     "recall_at_P" + str(TARGET_PREC): round(r_tp, 4),
                     "scrap_rate_all": round(float((s_all > float(tau)).mean()), 5)})

metrics = pd.DataFrame(rows)
csv_path = REEVAL_DIR / f"{VERSION}_0305_sec3_reweight_metrics_{EVAL_SPLIT}.csv"
metrics.to_csv(csv_path, index=False)
print("->", csv_path)
display(metrics.round(4))

# τ per tuning source, side by side — drift = tuned_oot − tuned: positive means the train-tuned
# cutoff sits below what the eval slice needs for precision >= target (over-scraps out of time)
tau_by_src = (metrics.loc[metrics["tau_source"] != "grid"]
              .pivot(index="model", columns="tau_source", values="tau"))
if {"tuned", "tuned_oot"} <= set(tau_by_src.columns):
    tau_by_src["drift"] = tau_by_src["tuned_oot"] - tau_by_src["tuned"]
display(tau_by_src.round(4))
tau_by_src.to_csv(REEVAL_DIR / f"{VERSION}_0305_sec3_tau_by_source_{EVAL_SPLIT}.csv")

In [ ]:
# §3b — company convention: the same operating points on the WHOLE eval split
# Allianz reads precision/recall against the recorded outcome of EVERY claim in the split —
# scrapped rows (label forced to 1) and rows the deciding log never treated included. §3 is the
# verified-slice reading; this is the business's. Side by side, the gap is the forced-label
# inflation itself: a scrapped row can only ever land as TP or FN, never FP.
def load_full_split(split: str) -> tuple[pd.DataFrame, list[str]]:
    """targets(split) — every row with its recorded label — plus one score column per model."""
    t = pd.read_parquet(config.split_path("targets", VERSION, split))[[ID_COL, "observed"]]
    return attach_scores(t, split)


ev_all, all_models = load_full_split(EVAL_SPLIT)
tr_all, tr_all_models = load_full_split(TRAIN_SPLIT)
assert all_models == MODELS, "score files differ between the treated and the full read — rerun §2"
ev_all = ev_all.merge(ev[[ID_COL, "decision"]], on=ID_COL, how="left")   # NaN = never treated
n_scr = int((ev_all["decision"] == 1).sum()); n_unt = int(ev_all["decision"].isna().sum())
print(f"eval whole split: {len(ev_all):,} rows = {len(ev_gar):,} garage + {n_scr:,} scrapped "
      f"(label forced to 1) + {n_unt:,} untreated | recorded TL: {int(ev_all['observed'].sum()):,}")

rows = []
for name in MODELS:
    g = ev_all[[ID_COL, "observed", "decision", name]].dropna(subset=[name])
    y = g["observed"].astype(int).to_numpy()
    s = g[name].to_numpy(dtype=float)
    forced = (g["decision"] == 1).to_numpy()
    psi = peak0(residual(y, s))
    r_tp = recall_at_precision(y, s, TARGET_PREC)

    # the §3 operating points (grid + garage-tuned), re-scored on the whole split …
    taus = [(float(t), src) for t, src in
            metrics.loc[metrics["model"] == name, ["tau", "tau_source"]].itertuples(index=False)]
    # … plus the cutoffs the company's own procedure would pick on the WHOLE split, forced labels
    # in — tuned_all on the train split, tuned_oot_all on the eval split itself (§3's twins)
    if TUNE_TAU:
        for src, (frame, avail) in {"tuned_all": (tr_all, tr_all_models),
                                    "tuned_oot_all": (ev_all, all_models)}.items():
            if name not in avail:
                continue
            tg = frame[["observed", name]].dropna(subset=[name])
            t = threshold.tune(tg["observed"].astype(int).to_numpy(),
                               tg[name].to_numpy(dtype=float), target_precision=TARGET_PREC)
            if t is None:
                print(f"  {name}: no cutoff reaches precision>={TARGET_PREC} on the whole split ({src})")
                continue
            taus.append((float(t), src))
            s_ev = ev[[ID_COL, name]].dropna(subset=[name])          # same row basis as §3's files
            dp = REEVAL_DIR / f"{VERSION}_decisions_{EVAL_SPLIT}_{name}_tau{round(float(t), 6)}.parquet"
            pd.DataFrame({ID_COL: s_ev[ID_COL].to_numpy(),
                          "score": s_ev[name].to_numpy(dtype=float),
                          "decision": (s_ev[name].to_numpy(dtype=float) > float(t)).astype(int)}
                         ).to_parquet(dp, index=False)

    for tau, src in taus:
        rows.append({"model": name, "tau": round(float(tau), 6), "tau_source": src,
                     "n_eval_all": len(g), **metrics_at(y, s, float(tau)),
                     "TP_forced": int(((s > float(tau)) & forced).sum()),
                     "psi_loop": round(psi, 3),
                     "recall_at_P" + str(TARGET_PREC): round(r_tp, 4),
                     "scrap_rate_all": round(float((s > float(tau)).mean()), 5)})

metrics_all = pd.DataFrame(rows)
csv_all = REEVAL_DIR / f"{VERSION}_0305_sec3b_reweight_metrics_{EVAL_SPLIT}_all.csv"
metrics_all.to_csv(csv_all, index=False)
print("->", csv_all)
display(metrics_all.round(4))

# garage vs whole split at the same (model, τ): precision_gap is the inflation the thesis reports
keys = ["model", "tau", "tau_source"]
side = (metrics[keys + ["precision", "recall"]]
        .merge(metrics_all[keys + ["precision", "recall", "TP_forced"]], on=keys,
               suffixes=("_garage", "_all")))
side["precision_gap"] = side["precision_all"] - side["precision_garage"]
display(side.round(4))
side.to_csv(REEVAL_DIR / f"{VERSION}_0305_sec3b_precision_gap_{EVAL_SPLIT}.csv", index=False)

# all four tuned cutoffs per model: garage vs whole split (forced-label shift) × train vs eval (drift)
tau_all_by_src = (metrics_all.loc[metrics_all["tau_source"] != "grid"]
                  .pivot(index="model", columns="tau_source", values="tau").round(4))
display(tau_all_by_src)
tau_all_by_src.to_csv(REEVAL_DIR / f"{VERSION}_0305_sec3b_tau_by_source_{EVAL_SPLIT}_all.csv")

In [ ]:
# §3c — does naive's REDUCED population reproduce baseline's real performance?
# naive is fit on corrector_targets — 03_01 already dropped rows the treatment source never
# covered (v3: claim_ids absent from the v2 log; v2: awaiting-authorisation/unrecovered/NaN
# statuses). If naive's performance at the SAME fixed cutoff differs a lot from "baseline" (the
# original deployed-or-attempted model, scored on the SAME rows), the dropped rows may have
# carried real information — not just noise safely discarded. This is a DIAGNOSTIC, not a
# correction: rarity/transport/pnu share naive's population, so a flag here questions the WHOLE
# naive-vs-corrected comparison for this version, not just naive on its own.
GAP_ALERT = {"precision": 0.02, "recall": 0.05}   # absolute; precision tighter (business-critical
                                                   # constraint is precision >= 0.985) — adjust
                                                   # and re-run if the real numbers warrant it

if "baseline" not in MODELS:
    print(f"§3c skipped — no baseline scores for {VERSION} (run src/scoring/score_all.py first)")
elif "naive" not in MODELS:
    print(f"§3c skipped — no naive scores for {VERSION} (run 03_02/03_03 first)")
else:
    def _pick(df: pd.DataFrame) -> pd.DataFrame:
        sub = df[(df["tau_source"] == "grid") & (df["model"].isin(["baseline", "naive"]))
                 & (np.abs(df["tau"] - HEADLINE_TAU) < 1e-6)]
        return sub.set_index("model")[["AUC", "precision", "recall", "TP", "FP", "FN", "TN"]]

    for label, tag, df in (("garage slice, thesis reading (§3)", "garage", metrics),
                           ("whole split, company convention (§3b)", "all", metrics_all)):
        pick = _pick(df)
        if not {"baseline", "naive"} <= set(pick.index):
            print(f"[{label}] missing baseline or naive row at tau={HEADLINE_TAU} — skipped")
            continue
        print(f"\n[{label}] baseline vs naive @ tau={HEADLINE_TAU} (grid):")
        display(pick.round(4))
        figstyle.save_table(pick.round(4), f"{VERSION}_0305_sec3c_naive_vs_baseline_{tag}")
        gap = pick.loc["naive"] - pick.loc["baseline"]
        print("  gap (naive - baseline): " +
              ", ".join(f"{m}={gap[m]:+.4f}" for m in ("AUC", "precision", "recall")))

        flags = [m for m in ("precision", "recall") if abs(gap[m]) > GAP_ALERT[m]]
        if flags:
            print(f"  !! gap exceeds the alert tolerance on {flags} "
                  f"({', '.join(f'{m}={gap[m]:+.4f}' for m in flags)}) — the rows 03_01 dropped "
                  f"for {VERSION} may carry real information, not just noise. State this "
                  f"explicitly as a limitation on the mitigation comparison, rather than reading "
                  f"rarity/transport/pnu's gain over naive as the full story.")
        else:
            print("  within the alert tolerance — no flag on this slice.")

In [ ]:
# §4 — PR trade-off on the eval garage slice (threshold swept), house style
fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for name in MODELS:
    g = ev_gar[["observed", name]].dropna(subset=[name])
    y = g["observed"].astype(int).to_numpy()
    prec, rec, _ = precision_recall_curve(y, g[name].to_numpy(dtype=float))
    if name == "baseline":
        ax.plot(rec[:-1], prec[:-1], ls=":", color=figstyle.NEUTRAL, label="baseline")
    else:
        ax.plot(rec[:-1], prec[:-1], label=name)     # colour cycle assigns slots in order
ax.axhline(TARGET_PREC, ls="--", lw=1, color=figstyle.INK)
ax.text(0.01, TARGET_PREC + 0.01, f"{TARGET_PREC} target", fontsize=8, color=figstyle.INK)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"{VERSION} reweight axes — PR on {EVAL_SPLIT} garage rows")
ax.legend(fontsize=8)
figstyle.save(fig, f"{VERSION}_0305_sec4_reweight_pr_{EVAL_SPLIT}")
plt.show()

In [ ]:
# §4b — PR trade-off on the WHOLE eval split (company convention), same style as §4
fig, ax = plt.subplots(figsize=figstyle.FIG_1)
for name in MODELS:
    g = ev_all[["observed", name]].dropna(subset=[name])
    y = g["observed"].astype(int).to_numpy()
    prec, rec, _ = precision_recall_curve(y, g[name].to_numpy(dtype=float))
    if name == "baseline":
        ax.plot(rec[:-1], prec[:-1], ls=":", color=figstyle.NEUTRAL, label="baseline")
    else:
        ax.plot(rec[:-1], prec[:-1], label=name)     # colour cycle assigns slots in order
ax.axhline(TARGET_PREC, ls="--", lw=1, color=figstyle.INK)
ax.text(0.01, TARGET_PREC + 0.01, f"{TARGET_PREC} target", fontsize=8, color=figstyle.INK)
ax.set_xlabel("recall"); ax.set_ylabel("precision"); ax.set_ylim(0, 1.02)
ax.set_title(f"{VERSION} reweight axes — PR on {EVAL_SPLIT}, whole split (forced labels in)")
ax.legend(fontsize=8)
figstyle.save(fig, f"{VERSION}_0305_sec4b_reweight_pr_{EVAL_SPLIT}_all")
plt.show()

## §5 — how to read this (and what it cannot say)

- **Precision is measured on the verified (garage) slice only.** The high-score region is
  almost empty of garage rows (positivity), so a τ that holds precision on the train slice can
  fail out-of-time — 03_02's synthetic study showed exactly that collapse. Read the *ranking*
  of axes and the `recall_at_P` column, not absolute levels.
- **§3b is the company's number, not a second truth.** On the whole split a scrapped row
  (label forced to 1) can only fall as TP or FN — never FP — so precision there is inflated by
  construction; `TP_forced` counts exactly those rows and `precision_gap` (§3b's last table) is
  the inflation at the same (model, τ). `tuned_all` is what the company's procedure would pick;
  its distance from `tuned` is how far the forced labels move the operating point. For v2 the
  untreated rows are the status-dropped groups (03_01 §4), which the company's own training
  swallowed — they are back in here, as in its convention.
- **grid / tuned / tuned_oot answer different questions**: grid = the production cutoff imposed
  on a rescaled score (reference only); tuned = each model's own comparable operating point,
  from `threshold.tune`'s precision-floor mode on the train slice — the company
  `select_best_threshold` rule, adopted 2026-09-02; tuned_oot = the same rule on the eval slice
  itself, so its precision ≥ target holds by construction. Neither is a clean held-out: train
  is the retrained model's own fitting data (in-sample for the fit), the eval slice is the
  evaluation itself (in-sample for the evaluation). Their τ gap (`drift` in §3) is the transfer
  loss — a positive drift means the train-tuned cutoff sits too low and over-scraps out of time.
  `recall_at_P` is the recall at tuned_oot's cutoff, whichever row is present.
- **§3c — naive vs baseline at `HEADLINE_TAU`, a population-loss diagnostic.** This is the only
  place a FIXED cutoff (not tuned) is used on purpose: 0.872 for v2, 0.984 for v3 (§1). A large
  gap here is not evidence one scheme beats another — it questions whether naive's population
  (03_01's drop-adjusted `corrector_targets`) is close enough to the real deployed/attempted
  model's population to trust naive as a baseline at all. If `GAP_ALERT` fires, say so
  explicitly wherever this version's mitigation results are reported — do not silently proceed
  as if the correction schemes are being compared on an uncontroversial baseline.
- ψ (`psi_loop`) is threshold-free and reads the loop signal, not deployability — a scheme can
  ease ψ and still fail the precision constraint.
- Rows the deciding log never scored have no treatment: excluded from garage metrics
  (`scrap_rate_all` still uses every scored row).
- **Channel (FNOL/ENOL) is an explicitly excluded confound, not an oversight.** `targets` and
  the treatment sources here carry no channel column and `src/schema.py` declares none
  canonical; a case-mix difference between channels (03_01's notes) is a plausible confound for
  any of these comparisons, but studying it needs a real column name off the company laptop and
  more time than this thesis has. Decision (2026-09-13), now also in the thesis at
  `sec:concl-status`: out of scope for the whole thesis — say so as a stated limitation wherever
  channel could plausibly matter, never silently.
- **File naming**: every CSV this notebook writes to `src/data/real/reeval/` carries `0305`
  (this notebook) and its section (`sec3`/`sec3b`/`sec3c`) in the filename; §1 redirects
  `figstyle.FIG_DIR` to `figures/mitigation/03_05/<VERSION>/` once `VERSION` is known (keyed on
  VERSION only — TRAIN_SPLIT and EVAL_SPLIT are both used together in one run, never an
  independent rerun knob), where §3c/§4/§4b's tables and figures land under the same `0305_secN`
  naming. Matches 03_01–03_04's convention.
- More than ~7 axes will exhaust the 8-slot colour cycle in §4 — fold or split the figure
  rather than inventing a 9th colour.